## Imports

In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests
from global_variables import LOCAL_MODEL_NAME, LOCAL_BASE_URL


## Globale Variablen

In [7]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig. Gib falls Informationen aus der Fuktion search_lecture_docs entnommen werden das heißt dass die Informationen aus einer Datei kommen, immer den Dateipfad mit an."

LOCAL = True

## Tools

Retreival Tool

In [8]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description='Retrieves information from Lecture related Documents')
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()

## Initialisierungen

In [9]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url=LOCAL_BASE_URL,
    model=LOCAL_MODEL_NAME,
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[search_lecture_docs], system_prompt=system_prompt)

In [10]:
prompt = str(input())
result = agent.invoke({"messages": [("user", prompt)]})
print(result["messages"][-1].content)

Ja – Mitarbeiter*innen können sich fachlich weiterbilden. Laut den bereitgestellten Unterlagen können sie jährlich bis zu **1 000 €** für fachliche Weiterbildungen beantragen.  

Quelle: `data/processed/example.md`
